# EDA Survei Kos Toba

Notebook ini menyelesaikan seluruh tugas Issue #15:
- distribusi harga bulanan normalisasi;
- harga berdasarkan kecamatan dan kategori penghuni;
- distribusi luas kamar dan lantai;
- hubungan fasilitas dengan harga;
- hubungan jarak fasilitas dengan harga;
- missing, outlier, dan status data;
- interpretasi untuk setiap grafik.

In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

DATASET_VERSION = 'survey_clean_v1'
SEED = 42
project = next(path for path in [Path.cwd(), *Path.cwd().parents]
               if (path / 'data/raw/data_fix.csv').exists())
raw_path = project / 'data/raw/data_fix.csv'
staging_path = project / 'data/staging/survey_clean_v1.csv'
warehouse_path = project / 'data/tobakost_survey_warehouse.db'
figure_dir = project / 'reports/figures/issue-15'
figure_dir.mkdir(parents=True, exist_ok=True)
plt.style.use('seaborn-v0_8-whitegrid')

raw = pd.read_csv(raw_path)
df = pd.read_csv(staging_path)
analysis = df.loc[df['accepted_for_analysis'].eq(1)].copy()

with sqlite3.connect(warehouse_path) as conn:
    warehouse_count = conn.execute('SELECT COUNT(*) FROM fact_sewa_kos').fetchone()[0]
    model_ready_count = conn.execute('SELECT COUNT(*) FROM fact_sewa_kos WHERE model_ready = 1').fetchone()[0]

print({'dataset_version': DATASET_VERSION, 'seed': SEED, 'raw_rows': len(raw),
       'staging_rows': len(df), 'analysis_rows': len(analysis),
       'warehouse_fact_rows': warehouse_count, 'model_ready_rows': model_ready_count})

## Ringkasan status data

In [ ]:
quality_summary = pd.DataFrame({
    'Status': ['Exact duplicate', 'Duplikasi logis', 'Kandidat outlier IQR',
               'Lolos analisis', 'Model ready'],
    'Jumlah': [int(df['exact_duplicate'].sum()), int(df['duplicate_logis'].sum()),
               int(df['price_anomaly_iqr'].sum()), int(df['accepted_for_analysis'].sum()),
               int(df['model_ready'].sum())]
})
quality_summary

## 1. Distribusi harga bulanan normalisasi

In [ ]:
prices = pd.to_numeric(analysis['harga_bulanan_norm'], errors='coerce').dropna()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(prices, bins=24, color='#2563eb', edgecolor='white', alpha=0.85)
axes[0].axvline(prices.median(), color='#dc2626', linestyle='--',
                label=f'Median Rp{prices.median():,.0f}')
axes[0].set(title='Histogram Harga Setara Bulanan', xlabel='Tarif bulanan (Rp)',
            ylabel='Jumlah unit')
axes[0].ticklabel_format(style='plain', axis='x')
axes[0].legend()
axes[1].boxplot(prices, vert=True, patch_artist=True,
                boxprops={'facecolor':'#dbeafe'})
axes[1].set(title='Boxplot Harga Setara Bulanan', ylabel='Tarif bulanan (Rp)')
axes[1].ticklabel_format(style='plain', axis='y')
fig.tight_layout()
fig.savefig(figure_dir/'01_distribusi_harga.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print({'median': prices.median(), 'q1': prices.quantile(.25), 'q3': prices.quantile(.75)})

**Interpretasi:** Median harga setara bulanan adalah Rp687.500.
Kuartil pertama Rp550.000 dan kuartil ketiga Rp833.333. Distribusi memiliki ekor
ke arah tarif tinggi; median lebih representatif daripada rata-rata untuk tarif umum.

## 2. Harga berdasarkan kecamatan dan kategori penghuni

In [ ]:
kec = analysis.groupby('kecamatan')['harga_bulanan_norm'].agg(['count','median']).sort_values('median')
category = analysis.groupby('kategori_penghuni')['harga_bulanan_norm'].agg(['count','median']).sort_values('median')
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].bar(kec.index, kec['median'], color='#2563eb')
axes[0].set(title='Median Harga per Kecamatan', xlabel='Kecamatan', ylabel='Median Rp/bulan')
axes[0].tick_params(axis='x', rotation=45)
axes[0].ticklabel_format(style='plain', axis='y')
axes[1].bar(category.index, category['median'], color='#14b8a6')
axes[1].set(title='Median Harga per Kategori Penghuni', xlabel='Kategori', ylabel='Median Rp/bulan')
axes[1].tick_params(axis='x', rotation=25)
axes[1].ticklabel_format(style='plain', axis='y')
fig.tight_layout()
fig.savefig(figure_dir/'02_harga_kecamatan_kategori.png', dpi=150, bbox_inches='tight')
plt.close(fig)
display(kec)
display(category)

**Interpretasi:** Median berbeda antar kecamatan. Balige dan Laguboti mempunyai
median relatif tinggi pada subset ini, sedangkan Sigumpar rendah. Perbedaan dapat
dipengaruhi jumlah sampel, lokasi, luas kamar, dan fasilitas. Grafik kategori penghuni
bersifat deskriptif dan tidak mengendalikan faktor lain.

## 3. Distribusi luas kamar dan lantai

In [ ]:
areas = pd.to_numeric(analysis['luas_kamar_m2'], errors='coerce').dropna()
floor = analysis.groupby('jumlah_lantai')['harga_bulanan_norm'].agg(['count','median']).sort_index()
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].hist(areas, bins=20, color='#8b5cf6', edgecolor='white')
axes[0].axvline(areas.median(), color='#dc2626', linestyle='--',
                label=f'Median {areas.median():.1f} m2')
axes[0].set(title='Distribusi Luas Kamar', xlabel='Luas kamar (m2)', ylabel='Jumlah unit')
axes[0].legend()
axes[1].bar(floor.index.astype(str), floor['median'], color='#06b6d4')
axes[1].set(title='Median Harga per Lantai', xlabel='Lantai', ylabel='Median Rp/bulan')
axes[1].ticklabel_format(style='plain', axis='y')
fig.tight_layout()
fig.savefig(figure_dir/'03_luas_dan_lantai.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('median luas:', areas.median())
display(floor)

**Interpretasi:** Mayoritas unit terkumpul pada rentang luas tertentu.
Lantai 1 mempunyai 227 baris dan median Rp650.000; lantai 2 mempunyai 82 baris
dan median Rp800.000. Lantai 3 dan lantai 9 hanya masing-masing satu baris sehingga
belum cukup untuk generalisasi, dan lantai 9 perlu ditinjau sebagai anomali.

## 4. Hubungan fasilitas dengan harga

In [ ]:
wifi_yes = analysis['wifi'].fillna('').astype(str).str.lower().str.contains('ada|tersedia')
km_yes = analysis['tipe_kamar_mandi'].fillna('').astype(str).str.lower().str.contains('dalam')
facility = pd.DataFrame({
    'Kelompok': ['AC', 'AC', 'Wi-Fi', 'Wi-Fi', 'Kamar mandi', 'Kamar mandi'],
    'Fasilitas': ['Ada AC', 'Tidak ada AC', 'Ada Wi-Fi', 'Tanpa Wi-Fi', 'KM dalam', 'KM luar/lainnya'],
    'Median': [
        analysis.loc[analysis['ac'].eq(1), 'harga_bulanan_norm'].median(),
        analysis.loc[analysis['ac'].eq(0), 'harga_bulanan_norm'].median(),
        analysis.loc[wifi_yes, 'harga_bulanan_norm'].median(),
        analysis.loc[~wifi_yes, 'harga_bulanan_norm'].median(),
        analysis.loc[km_yes, 'harga_bulanan_norm'].median(),
        analysis.loc[~km_yes, 'harga_bulanan_norm'].median(),
    ],
})
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(facility['Fasilitas'], facility['Median'], color=['#10b981','#94a3b8','#10b981','#94a3b8','#10b981','#94a3b8'])
ax.set(title='Median Tarif Berdasarkan Fasilitas', xlabel='Fasilitas', ylabel='Median Rp/bulan')
ax.tick_params(axis='x', rotation=20)
ax.ticklabel_format(style='plain', axis='y')
fig.tight_layout()
fig.savefig(figure_dir/'04_fasilitas_vs_harga.png', dpi=150, bbox_inches='tight')
plt.close(fig)
facility

**Interpretasi:** Median fasilitas AC, Wi-Fi, dan kamar mandi dalam dibandingkan
dengan kelompok pembandingnya. Perbedaan menunjukkan hubungan deskriptif, bukan
nilai tambahan sebab-akibat karena lokasi, luas, lantai, dan fitur lain belum dikendalikan.

## 5. Hubungan jarak fasilitas dengan harga

In [ ]:
distance_columns = [column for column in analysis.columns if column.startswith('jarak_')]
distance_corr = {}
for column in distance_columns:
    pair = pd.DataFrame({'harga': pd.to_numeric(analysis['harga_bulanan_norm'], errors='coerce'),
                         'jarak': pd.to_numeric(analysis[column], errors='coerce')}).dropna()
    distance_corr[column] = pair.corr(method='spearman').iloc[0,1] if len(pair) >= 2 else 0
corr = pd.Series(distance_corr).sort_values()
labels = [name.replace('jarak_','').replace('_m','').replace('_',' ').title() for name in corr.index]
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(labels, corr.values, color='#6366f1')
ax.axvline(0, color='#0f172a', linewidth=.8)
ax.set(title='Korelasi Spearman Jarak Fasilitas vs Harga', xlabel='Koefisien korelasi')
fig.tight_layout()
fig.savefig(figure_dir/'05_jarak_vs_harga.png', dpi=150, bbox_inches='tight')
plt.close(fig)
corr.rename(index=dict(zip(corr.index, labels)))

**Interpretasi:** Korelasi Spearman digunakan untuk menangkap hubungan monoton.
Nilai mendekati nol berarti jarak fasilitas tidak memiliki hubungan monotonic kuat
dengan tarif pada sampel. Nilai negatif atau positif harus dibaca bersama lokasi dan
karakter unit; korelasi bukan bukti sebab-akibat.

## 6. Missing, outlier, dan status data

In [ ]:
missing = df.isna().sum().drop(labels=['rejection_reason'], errors='ignore').sort_values(ascending=False)
missing = missing[missing.gt(0)].head(8)
q1 = prices.quantile(.25)
q3 = prices.quantile(.75)
lower = q1 - 1.5 * (q3-q1)
upper = q3 + 1.5 * (q3-q1)
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].boxplot(prices, vert=True, patch_artist=True, boxprops={'facecolor':'#dbeafe'})
axes[0].axhline(lower, color='#dc2626', linestyle='--', label=f'Batas bawah {lower:,.0f}')
axes[0].axhline(upper, color='#dc2626', linestyle='--', label=f'Batas atas {upper:,.0f}')
axes[0].set(title='Outlier Harga IQR', ylabel='Rp/bulan')
axes[0].ticklabel_format(style='plain', axis='y')
axes[0].legend()
axes[1].barh(missing.index, missing.values, color='#f97316')
axes[1].set(title='Kolom dengan Missing Value', xlabel='Jumlah baris kosong')
axes[1].invert_yaxis()
fig.tight_layout()
fig.savefig(figure_dir/'06_missing_outlier_status.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print({'iqr_lower': lower, 'iqr_upper': upper, 'outlier_rows': int(df['price_anomaly_iqr'].sum())})
display(missing.rename('missing'))

**Interpretasi:** Ada 25 kandidat outlier IQR dan 30 baris terlibat duplikasi logis.
Keduanya ditandai pada staging, bukan langsung dihapus. Kolom
`tahun_mulai_menyewa` kosong pada seluruh baris karena sumber saat ini tidak
menyediakannya. Missing tidak diimputasi sembarang; baseline memakai `model_ready = 1`.

## Kesimpulan

1. Gunakan `harga_bulanan_norm` untuk membandingkan periode sewa berbeda.
2. Gunakan median untuk ringkasan tarif per kecamatan dan fasilitas.
3. Pertahankan flag duplikasi dan outlier untuk audit.
4. Tinjau lantai 9 dan baris kandidat outlier sebelum pemakaian lanjutan.
5. Gunakan subset `model_ready = 1` untuk baseline model konservatif.